# 03. Guardrails and Human in the Loop

This notebook shows how to validate outputs and route uncertain cases to a person. That is a core pattern for research workflows where ambiguity matters. The SDK README groups this under guardrails and human in the loop: guardrails check whether the run is safe or valid, and human review steps let a person intervene when the model should not decide alone.

## Why this matters in DH
In digital humanities, uncertainty is not always a failure. It can be part of the evidence. A blurred place name, a conflicting date, or a doubtful transcription often needs scholarly judgment. This notebook shows how to preserve that judgment instead of hiding it behind automation.

The goal is not to eliminate ambiguity. The goal is to make ambiguity visible and actionable.

## Concepts
- Guardrails
- Human review
- Uncertainty thresholds
- Safer automation
- Validation before action
- Evidence-based decision making

Relevant docs: [Guardrails](https://openai.github.io/openai-agents-python/guardrails/) and [Human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/).

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
The notebook uses a small set of ambiguous archival-style records stored in `data/` and loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Human in the loop: https://openai.github.io/openai-agents-python/human_in_the_loop/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [19]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, trace, set_tracing_export_api_key


In [20]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

records = [{'id': item['id'], 'text': item['text']} for item in load_documents()]
records

[{'id': 1,
  'text': 'Madrid, 4 April 1871\n\nDear brother,\n\nI have finally found a calm hour to write after the commotion of the exhibition. The galleries were crowded, and several visitors asked about the catalog, which made the day long but worthwhile.\n\nIn 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel. She also said she hoped the spring weather would make the journey easier when she next visited.\n\nWith affection,\nMaria'},
 {'id': 2,
  'text': 'Seville, 18 June 1894\n\nEsteemed colleague,\n\nThe printer in Seville announced a new edition of poems by Antonio Ruiz and asked readers to send subscriptions. The notice was meant to reach both bookshops and local reading circles, since the editor believed the edition might sell quickly.\n\nPlease advise whether the notice should also be sent to the archive copy desk. I have also enclosed a short list of the proofs that still require attention before the issue goes out.\n\n

In [21]:
@dataclass
class ReviewDecision:
    record_id: int
    uncertain: bool
    confidence: float
    notes: str


## Step 1: A tiny review heuristic

We treat low confidence or uncertain OCR as a signal for human review. The heuristic is intentionally simple so the policy is clear before a more complex validation setup.

In [22]:
def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

sample_decision = ReviewDecision(record_id=3, uncertain=True, confidence=0.52, notes='OCR ambiguity around place name')
should_review(sample_decision)

True

## Step 2: Build a guardrail

A guardrail can reject inputs that clearly need human review before the agent continues. In the SDK, guardrails are a first-class concept rather than an afterthought, which makes them useful for real workflows where you need a policy boundary before the model acts.

You can think of them as automated quality checks that sit in front of or behind the agent.

In [23]:
def uncertainty_guardrail(ctx, agent, input_text):
    if isinstance(input_text, str) and ('may be' in input_text.lower() or 'unclear' in input_text.lower()):
        return GuardrailFunctionOutput(output_info='uncertain text', tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
guardrail

InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x7b9dd05d8d60>, name='uncertainty_guardrail', run_in_parallel=True)

## Step 3: Agent with guardrail

If the text looks uncertain, the agent should stop and let the person review it. This is the main classroom lesson: a good agentic pipeline knows when not to proceed.

For research settings, that protects against fabricated precision and makes the workflow more trustworthy.

In [24]:
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
)

review_agent

Agent(name='Review Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Assess whether the text can be safely processed. If it is uncertain, return a short note asking for human review.', prompt=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x7b9dd05d8d60>, name='uncertainty_guardrail', run_in_parallel=True)], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm

## Step 4: Run on a clean and an uncertain record

First, run a clean record that should pass the guardrail. In notebooks, use `await Runner.run(...)` rather than `run_sync(...)` because the kernel already manages an event loop.

Then run an ambiguous record from the dataset to show the review path.

In [25]:
clean_text = records[0]['text']
with trace('DH guardrails clean case'):
    clean_result = await Runner.run(review_agent, clean_text)
print(clean_result.final_output)

Safe to process.


## Human review

If the next example triggers the guardrail, the right response is to review the record instead of forcing a guess.

In [26]:
uncertain_text = records[-1]['text']
try:
    with trace('DH guardrails uncertain case'):
        uncertain_result = await Runner.run(review_agent, uncertain_text)
    print(uncertain_result.final_output)
except Exception as exc:
    print(type(exc).__name__, exc)

Needs human review: the text is partially blurred and the reading/transcription is uncertain.


## Human-in-the-loop checkpoint

If the guardrail triggers, the next step is not to force a guess. It is to ask a person to resolve the ambiguity and then resume.

## Exercise

Change the threshold so only records with both OCR ambiguity and low confidence are sent to review.

## Solution

Modify `uncertainty_guardrail` to check for two conditions at once, for example `('unclear' in text and confidence < 0.8)`.